# Part 2 - Brick Detector Training

Fine-tuning YOLOv8 to find every LEGO brick in a photo. One class (`brick`); identifying the part is P2's job.
This notebook uses the dataset built by `01_build_dataset.ipynb`.

1. **Train:** start from the COCO-pretrained YOLOv8n weights and fine-tune all layers at 1280 x 1280, using the default YOLO box and classification loss.
2. **Pick the confidence threshold on dev:** the threshold with the best F1 on the dev photos.
3. **Final evaluation on test (once):** mAP@0.5 and recall on the held-out test photos, plus how often the brick count is exactly right.

Output for each image: an N x 6 array of `(x1, y1, x2, y2, confidence, class)` from `detector.detect()`.

## Settings

In [ ]:
from pathlib import Path

RUN_NAME = "yolov8n_1280"
MODEL = "yolov8n.pt"        # COCO pretrained, the fastest YOLOv8. yolov8s.pt is about 2x slower and slightly more accurate
IMGSZ = 1280
EPOCHS = 30                 # starting from COCO weights with one class, it converges well before this
PATIENCE = 8                # stop early if dev mAP hasn't improved for this many epochs
BATCH = 16
CACHE = "disk"              # decode each photo once and reuse it every epoch, instead of re-reading the JPEG
SEED = 539
SMOKE_TEST = False          # True: 10% of the training photos, 1 epoch at 640, to check everything runs

ULTRALYTICS = "ultralytics==8.4.216"
REPO_URL = "https://github.com/divijsinghkohli/cs539-backprop-boys.git"
WORK = Path("/content")
DATA_DIR = WORK / "lego_det"                     # images unzipped on the Colab VM
REPO_DIR = WORK / "repo"
DRIVE_DIR = Path("/content/drive/MyDrive/ece539-p1")
DATA_ZIP = DRIVE_DIR / "data" / "lego_det.zip"   # from 01_build_dataset
SPLITS_CSV = DRIVE_DIR / "results" / "splits.csv"

## Setup

Install ultralytics, mount the Drive, and get detector.py from GitHub

In [ ]:
get_ipython().system(f"pip install -q {ULTRALYTICS}")

import importlib, json, shutil, subprocess, sys
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "P1") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "P1"))
import detector
importlib.reload(detector)

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == 0 else "CPU (this will be very slow, switch the runtime to GPU)")

RUN = RUN_NAME + ("_smoke" if SMOKE_TEST else "")
imgsz = 640 if SMOKE_TEST else IMGSZ
CKPT_ROOT = DRIVE_DIR / "checkpoints"            # ultralytics writes weights/last.pt and best.pt here every epoch
RUN_DIR = DRIVE_DIR / "results" / "runs" / RUN   # small result files
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run {RUN}")

## Load the images

Copies `lego_det.zip` from Drive to the Colab VM, unzips it and writes the `data.yaml` file ultralytics reads.

In [ ]:
import zipfile

done_marker = DATA_DIR / ".unzipped"
if not done_marker.exists():
    local_zip = WORK / "lego_det.zip"
    shutil.copy(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(DATA_DIR)
    local_zip.unlink()
    done_marker.touch()

DATA_YAML = DATA_DIR / "data.yaml"
DATA_YAML.write_text(detector.data_yaml(DATA_DIR))

splits = pd.read_csv(SPLITS_CSV, dtype={"stem": str})
for split in ("train", "dev", "test"):
    n = len(list((DATA_DIR / "images" / split).glob("*.jpg")))
    assert n == (splits.split == split).sum(), f"{split}: {n} images on disk but splits.csv lists {(splits.split == split).sum()}"
    print(f"{split:5s}: {n} photos")

## Train

All layers are trained (`freeze=None`). Ultralytics saves `last.pt` and `best.pt` to Drive after every epoch, and keeps the weights with the best dev mAP as `best.pt`.
If the Colab session disconnects, re-run the notebook from the top and this cell picks up from `last.pt`.

After every epoch it prints an overall progress line with the time left. Expect roughly 30-45 minutes on a T4, less if it stops early.

In [ ]:
import os
import time
from ultralytics import YOLO

def show_progress(trainer):
    done = trainer.epoch + 1
    per_epoch = (time.time() - trainer.train_time_start) / (done - trainer.start_epoch)
    left = per_epoch * (trainer.epochs - done)
    filled = round(30 * done / trainer.epochs)
    print(f"Epoch {done}/{trainer.epochs} [{'=' * filled:<30}] {done / trainer.epochs:.0%} | "
          f"{per_epoch / 60:.1f} min/epoch | ~{left / 60:.0f} min left (less if it stops early)")

save_dir = CKPT_ROOT / RUN
last = save_dir / "weights" / "last.pt"

if last.exists():
    print(f"Resuming {RUN} from {last}")
    model = YOLO(str(last))
    model.add_callback("on_train_epoch_end", show_progress)
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.add_callback("on_train_epoch_end", show_progress)
    model.train(data=str(DATA_YAML), epochs=1 if SMOKE_TEST else EPOCHS, imgsz=imgsz, batch=BATCH,
                patience=PATIENCE, seed=SEED, freeze=None, pretrained=True,
                fraction=0.1 if SMOKE_TEST else 1.0, cache=CACHE, device=device, workers=os.cpu_count(),
                project=str(CKPT_ROOT), name=RUN, exist_ok=True, plots=True)

best_weights = save_dir / "weights" / "best.pt"
print("Best weights:", best_weights)

## Training curves

Loss, dev mAP@0.5 and dev recall per epoch from ultralytics' `results.csv`. Saved as `curves.png`.

In [ ]:
import matplotlib.pyplot as plt

log = pd.read_csv(save_dir / "results.csv")
log.columns = log.columns.str.strip()
log.to_csv(RUN_DIR / "log.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for loss in ("box_loss", "cls_loss", "dfl_loss"):
    axes[0].plot(log.epoch, log[f"train/{loss}"], label=f"train {loss}")
    axes[0].plot(log.epoch, log[f"val/{loss}"], linestyle="--", label=f"dev {loss}")
axes[0].set_title("Loss")
axes[1].plot(log.epoch, log["metrics/mAP50(B)"], marker="o", label="mAP@0.5")
axes[1].plot(log.epoch, log["metrics/mAP50-95(B)"], marker="o", label="mAP@0.5:0.95")
axes[1].set_title("Dev mAP")
axes[2].plot(log.epoch, log["metrics/recall(B)"], marker="o", label="recall")
axes[2].plot(log.epoch, log["metrics/precision(B)"], marker="o", label="precision")
axes[2].set_title("Dev precision / recall")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
fig.suptitle(RUN)
fig.tight_layout()
fig.savefig(RUN_DIR / "curves.png", dpi=150)
plt.show()

## Pick the confidence threshold on dev

Runs the best weights on every dev photo with a very low threshold, then checks precision, recall, F1 and exact count accuracy at thresholds from 0.05 to 0.95.
A detection counts as correct if it overlaps an unmatched true box with IoU >= 0.5. The threshold with the highest F1 is used for the test set and later by the pipeline.

In [ ]:
from PIL import Image

model = detector.load_model(best_weights)

def predictions_for(split, chunk=16):
    rows = splits[splits.split == split]
    paths = [str(DATA_DIR / "images" / split / f"{s}.jpg") for s in rows.stem]
    pairs = []
    for start in range(0, len(paths), chunk):
        batch = paths[start:start + chunk]
        dets = detector.detect(model, batch, conf=0.01, imgsz=imgsz, device=device)
        for path, d in zip(batch, dets):
            with Image.open(path) as img:
                w, h = img.size
            gt = detector.read_yolo_labels(Path(path.replace("/images/", "/labels/")).with_suffix(".txt"), w, h)
            pairs.append((d, gt))
    return rows.reset_index(drop=True), pairs

thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
dev_rows, dev_pairs = predictions_for("dev")
dev_table = pd.DataFrame(detector.threshold_table(dev_pairs, thresholds))
dev_table.to_csv(RUN_DIR / "dev_thresholds.csv", index=False)

CONF = float(dev_table.loc[dev_table.f1.idxmax(), "conf"])
print(f"Chosen confidence threshold: {CONF}")
dev_table.round(3)

## Final evaluation on test

Only run this once the settings are final. It reports:
* **mAP@0.5** (and mAP@0.5:0.95) from ultralytics' own evaluator
* **Precision, recall and count accuracy** at the threshold chosen on dev, for all test photos and split by single vs multi-brick photos

In [ ]:
val = model.val(data=str(DATA_YAML), split="test", imgsz=imgsz, batch=BATCH, device=device,
                project=str(RUN_DIR), name="test_val", exist_ok=True, plots=True, verbose=False)

test_rows, test_pairs = predictions_for("test")
overall = detector.threshold_table(test_pairs, [CONF])[0]
single = detector.threshold_table([p for p, n in zip(test_pairs, test_rows.n_bricks) if n == 1], [CONF])[0]
multi = detector.threshold_table([p for p, n in zip(test_pairs, test_rows.n_bricks) if n > 1], [CONF])[0]

results = {
    "run": RUN, "model": MODEL, "imgsz": imgsz, "epochs_trained": int(log.epoch.max()),
    "conf_threshold": CONF, "iou_threshold": 0.5,
    "test_photos": len(test_rows), "test_bricks": int(test_rows.n_bricks.sum()),
    "mAP50": float(val.box.map50), "mAP50_95": float(val.box.map),
    "precision": overall["precision"], "recall": overall["recall"], "f1": overall["f1"],
    "count_accuracy": overall["count_accuracy"],
    "single_brick_photos": {k: single[k] for k in ("recall", "precision", "count_accuracy")},
    "multi_brick_photos": {k: multi[k] for k in ("recall", "precision", "count_accuracy")},
}
(RUN_DIR / "test_metrics.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

## Example predictions

Green = true boxes, red = detections at the chosen threshold. The worst photos (most missed or extra bricks) are shown first, so failures are easy to spot. Saved to `samples/`.

In [ ]:
errors = [abs(int((d[:, 4] >= CONF).sum()) - len(gt)) for d, gt in test_pairs]
worst = np.argsort(errors)[::-1][:6]
show = list(dict.fromkeys(list(worst) + list(test_rows[test_rows.n_bricks > 1].index)))[:12]

(RUN_DIR / "samples").mkdir(exist_ok=True)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, i in zip(axes.flat, show):
    stem = test_rows.stem[i]
    img = Image.open(DATA_DIR / "images" / "test" / f"{stem}.jpg")
    d, gt = test_pairs[i]
    drawn = detector.draw_boxes(img, d[d[:, 4] >= CONF], gt)
    drawn.save(RUN_DIR / "samples" / f"{stem}.jpg", quality=90)
    ax.imshow(drawn)
    ax.set_title(f"{stem}: {int((d[:, 4] >= CONF).sum())} found / {len(gt)} true", fontsize=8)
    ax.axis("off")
fig.tight_layout()
fig.savefig(RUN_DIR / "test_examples.png", dpi=120)
plt.show()

## Export the result files

Results at: `MyDrive/ece539-p1/results/runs/<run>/`. The weights stay on Drive at `MyDrive/ece539-p1/checkpoints/<run>/weights/best.pt`.

In [ ]:
from google.colab import files

shutil.copy(save_dir / "args.yaml", RUN_DIR / "train_args.yaml")
archive = shutil.make_archive(str(WORK / f"{RUN}_results"), "zip", RUN_DIR)
print("Results:", sorted(p.name for p in RUN_DIR.iterdir()))
files.download(archive)